# CBR Legal Case Retrieval System

Nama : Ach Sofyan Daynur
NIM : 202110370311155

Implementasi Case-Based Reasoning (CBR) untuk Retrieval Kasus Hukum Menggunakan TF-IDF dan Cosine Similarity

# 02 Case Representation

## CBR Legal Case Retrieval System

Tahap ini bertujuan untuk membangun representasi kasus hukum dari dokumen putusan.

Metadata yang diekstraksi:

- Nomor Putusan
- Tahun
- Pengadilan
- Jenis Perkara
- Pihak
- Pasal
- Ringkasan Fakta

Hasil akhir disimpan sebagai representasi kasus yang siap digunakan pada proses retrieval.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!rm -rf /content/CBR-Legal-Case
!cp -r "/content/drive/MyDrive/CBR-Legal-Case" /content/

In [4]:
%cd /content/CBR-Legal-Case

/content/CBR-Legal-Case


Cell 2 — Import Library

In [7]:
import re
import pandas as pd

Cell 3 — Load Dataset

In [8]:
df = pd.read_csv(
    "/content/CBR-Legal-Case/data/processed/cases.csv"
)

print("Jumlah kasus:", len(df))
df.head()

Jumlah kasus: 30


,case_id,text,nomor_putusan,tahun,pengadilan,jenis_perkara,pihak,pasal,ringkasan_fakta
0,putusan 11,hkama\nahkamah Agung Repub\nahkamah Agung Repu...,961/PID.SUS/2026/PT,2026,Pengadilan Tinggi Surabaya,Pidana Khusus,MOHAMMAD BEDRUN BIN SAMHERI,"Pasal 609, Pasal 114, pasal 114, pasal 14",Terdakwa diajukan ke persidangan oleh Kejaksaa...
1,putusan 8,hkama\nahkamah Agung Repub\nahkamah Agung Repu...,296/PID.SUS/2026/PT,2026,Tidak ditemukan,Pidana Khusus,ISKANDAR AK BIN ABU KOSIM,"Pasal 609, Pasal 25, pasal 80, Pasal\n609, Pas...","Penahanan oleh: 1. Penyidik, sejak tanggal 3 S..."
2,putusan 28,hkama\nahkamah Agung Repub\nahkamah Agung Repu...,99/Pid,Tidak ditemukan,Tidak ditemukan,Tidak diketahui,Tidak ditemukan,"Pasal\n114, Pasal 132, Pasal 112",Terdakwa diajukan ke persidangan oleh Penuntut...
3,putusan 20,hkama\nahkamah Agung Repub\nahkamah Agung Repu...,53/,Tidak ditemukan,Tidak ditemukan,Pidana Khusus,Tidak ditemukan,"Pasal 112, Pasal 127, Pasal 114","17 Desa Batu Mareah, Kecamatan Sirimau, Kota A..."
4,putusan 18,hkama\nahkamah Agung Repub\nahkamah Agung Repu...,2281,Tidak ditemukan,Tidak ditemukan,Pidana Khusus,Tidak ditemukan,"Pasal\n114, Pasal 132, Pasal 127, Pasal\n112",a : Islam ; Pekerjaan : Wiraswasta ; Terdakwa ...


Cell 4 — Informasi Dataset


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   case_id          30 non-null     object
 1   text             30 non-null     object
 2   nomor_putusan    30 non-null     object
 3   tahun            30 non-null     object
 4   pengadilan       30 non-null     object
 5   jenis_perkara    30 non-null     object
 6   pihak            30 non-null     object
 7   pasal            30 non-null     object
 8   ringkasan_fakta  30 non-null     object
dtypes: object(9)
memory usage: 2.2+ KB


Cell 5 — Ekstraksi Nomor Putusan

In [10]:
def extract_nomor_putusan(text):

    match = re.search(
        r'Nomor\s+([A-Za-z0-9./-]+)',
        str(text),
        re.IGNORECASE
    )

    if match:
        return match.group(1)

    return "Tidak ditemukan"


df["nomor_putusan"] = df["text"].apply(
    extract_nomor_putusan
)

df[["case_id","nomor_putusan"]].head()

,case_id,nomor_putusan
0,putusan 11,961/PID.SUS/2026/PT
1,putusan 8,296/PID.SUS/2026/PT
2,putusan 28,99/Pid
3,putusan 20,53/
4,putusan 18,2281


Cell 6 — Ekstraksi Tahun

In [11]:
def extract_tahun(text):

    match = re.search(
        r'20\d{2}',
        str(text)
    )

    if match:
        return match.group()

    return "Tidak ditemukan"


df["tahun"] = df["text"].apply(
    extract_tahun
)

df[["case_id","tahun"]].head()

,case_id,tahun
0,putusan 11,2026
1,putusan 8,2026
2,putusan 28,2019
3,putusan 20,2021
4,putusan 18,2016


Cell 7 — Ekstraksi Pengadilan

In [12]:
def extract_pengadilan(text):

    match = re.search(
        r'Pengadilan\s+Tinggi\s+[A-Za-z]+',
        str(text)
    )

    if match:
        return match.group()

    return "Tidak ditemukan"


df["pengadilan"] = df["text"].apply(
    extract_pengadilan
)

df[["case_id","pengadilan"]].head()

,case_id,pengadilan
0,putusan 11,Pengadilan Tinggi Surabaya
1,putusan 8,Pengadilan Tinggi Palembang
2,putusan 28,Pengadilan Tinggi Medan
3,putusan 20,Pengadilan Tinggi Ambon
4,putusan 18,Pengadilan \nTinggi\nYogyakarta


dan tampil 5 baris pertama.

Cell 8 — Ekstraksi Jenis Perkara

In [13]:
def extract_jenis_perkara(text):

    text = str(text).lower()

    if "pid.sus" in text:
        return "Pidana Khusus"

    elif "pid" in text:
        return "Pidana"

    return "Tidak diketahui"


df["jenis_perkara"] = df["text"].apply(
    extract_jenis_perkara
)

df[["case_id","jenis_perkara"]].head()

,case_id,jenis_perkara
0,putusan 11,Pidana Khusus
1,putusan 8,Pidana Khusus
2,putusan 28,Pidana Khusus
3,putusan 20,Pidana Khusus
4,putusan 18,Pidana Khusus


Cell 9 — Ekstraksi Nama Terdakwa (Pihak)

In [14]:
import re

def clean_text(text):
    text = str(text)

    # OCR fix: "N a m a" → "Nama"
    text = re.sub(r'(?<=\w)\s(?=\w)', '', text)

    # rapikan spasi
    text = re.sub(r'\s+', ' ', text)

    return text


def extract_candidates(text):
    text = clean_text(text)

    # ambil semua kandidat nama besar (huruf kapital panjang)
    candidates = re.findall(r'([A-Z][A-Z\s\.]{8,})', text)

    blacklist = [
        "PUTUSAN", "DEMI", "KEADILAN", "TERDAKWA",
        "NAMA", "LENGKAP", "TEMPAT", "LAHIR"
    ]

    clean_candidates = []

    for c in candidates:
        c = re.sub(r'\s+', ' ', c).strip()

        if not any(b in c for b in blacklist):
            clean_candidates.append(c)

    return clean_candidates


def pick_main_defendant(text):
    text = clean_text(text)

    # PRIORITAS 1: setelah kata Nama / Terdakwa
    match = re.search(
        r'(Nama lengkap|Namalengkap|Nama|Terdakwa)\s*:?\s*(.*)',
        text,
        re.IGNORECASE
    )

    if match:
        kandidat = match.group(2)

        kandidat = re.split(r';|\n|Tempat|Umur|Agama|Pekerjaan', kandidat)[0]

        kandidat = re.sub(r'(alias|als|bin|binti)', ' ', kandidat, flags=re.IGNORECASE)
        kandidat = re.sub(r'[^A-Za-z\s]', ' ', kandidat)
        kandidat = re.sub(r'\s+', ' ', kandidat).strip()

        if len(kandidat) > 5:
            return kandidat

    # PRIORITAS 2: fallback dari kandidat terbesar
    cands = extract_candidates(text)

    if cands:
        return cands[0]  # ambil kandidat paling awal (biasanya terdakwa utama)

    return "Tidak ditemukan"


df["terdakwa"] = df["text"].apply(pick_main_defendant)

df[["case_id", "terdakwa"]]

,case_id,terdakwa
0,putusan 11,Namalengkap MOHAMMADBEDRUN SAMHERI
1,putusan 8,ISKANDARAKBINABUKOSIM
2,putusan 28,Terdakwa
3,putusan 20,Nama JAMALTUAKIA MALO DELON
4,putusan 18,Nama ABDULAZIS ANDI H SULAIMAN
5,putusan 24,Nama SUSANDHI SUKATMA AAN
6,PUTUSAN 3,RATNANAMZAH yangjugasebagaisalahsatuAnggotaDPR...
7,PUTUSAN 2,Nama AMSUR ANCU SANUSI Alm
8,putusan 10,Namalengkap FERYIRAWANSEMBIRING
9,putusan 15,TerdakwaI Namalengkap Salihuddin Udin Dehan


Cell 10 — Ekstraksi Pasal

In [15]:
def extract_pasal(text):

    pasal = re.findall(
        r'Pasal\s+\d+',
        str(text),
        re.IGNORECASE
    )

    if len(pasal) > 0:
        return ", ".join(
            sorted(set(pasal))
        )

    return "Tidak ditemukan"


df["pasal"] = df["text"].apply(
    extract_pasal
)

df[["case_id","pasal"]].head()

,case_id,pasal
0,putusan 11,"Pasal 114, Pasal 609, pasal 240, pasal 609, ..."
1,putusan 8,"Pasal\n609, Pasal 114, Pasal 25, Pasal 609, pa..."
2,putusan 28,"Pasal\n112, Pasal\n114, Pasal 112, Pasal 127, ..."
3,putusan 20,"Pasal\n127, Pasal 112, Pasal 127, Pasal 112,..."
4,putusan 18,"Pasal\n112, Pasal\n114, Pasal\n13, Pasal\n253,..."


Cell 11 — Ringkasan Fakta

In [16]:
import re

def clean_text(text):
    text = str(text)

    # hapus noise OCR header
    text = re.sub(r'hkama|ahkamah Agung.*?Indonesia', ' ', text, flags=re.IGNORECASE)

    # normalisasi spasi
    text = re.sub(r'\s+', ' ', text)

    return text


def extract_ringkasan_fakta(text):
    text = clean_text(text)

    # =========================
    # ambil bagian setelah kata kunci penting
    # =========================
    match = re.search(
        r'(Menimbang|Bahwa|Fakta|Duduk Perkara|Pertimbangan).*',
        text,
        re.IGNORECASE
    )

    if match:
        fakta = match.group(0)

        # stop di bagian hukum lain
        fakta = re.split(
            r'(MENGADILI|PUTUSAN|AMAR PUTUSAN|Demikian diputuskan)',
            fakta,
            flags=re.IGNORECASE
        )[0]

        # bersihkan noise
        fakta = re.sub(r'ahkamah.*?Repub.*?Indonesia', ' ', fakta, flags=re.IGNORECASE)
        fakta = re.sub(r'\s+', ' ', fakta).strip()

        return fakta

    # fallback: ambil sebagian teks tengah
    return text[:2000]


df["ringkasan_fakta"] = df["text"].apply(extract_ringkasan_fakta)

df[["case_id", "ringkasan_fakta"]]

,case_id,ringkasan_fakta
0,putusan 11,bahwa terdakwa telah mengajukan permintaan ban...
1,putusan 8,bahwa pada tanggal 23 April 2026 Penuntut Umum...
2,putusan 28,"Menimbang, bahwa Terdakwa-Terdakwa diajukan ke..."
3,putusan 20,Bahwa Ia Terdakwa JAMAL TUAKIA Alias MALO Alia...
4,putusan 18,Bahwa Terdakwa Abdul Azis alias Andi bin H. Su...
5,putusan 24,"Bahwa la, Terdakwa SUSANDHI BIN SUKATMA alias ..."
6,PUTUSAN 3,"DUDUK PERKARANYA : Menimbang, bahwa Pemohon Pr..."
7,PUTUSAN 2,Bahwa Terdakwa AMSUR Bin SANUSI (Alm) dan Saud...
8,putusan 10,bahwa Terdakwa diajukan didepan persidangan Pe...
9,putusan 15,"Menimbang, bahwa Para Terdakwa diajukan ke per..."


Cell 12 — Hasil Representasi Kasus

In [19]:
df[
    [
        "case_id",
        "nomor_putusan",
        "tahun",
        "pengadilan",
        "jenis_perkara",
        "pihak",
        "pasal",
        "ringkasan_fakta"
    ]
].head()

,case_id,nomor_putusan,tahun,pengadilan,jenis_perkara,pihak,pasal,ringkasan_fakta
0,putusan 11,961/PID.SUS/2026/PT,2026,Pengadilan Tinggi Surabaya,Pidana Khusus,MOHAMMAD BEDRUN BIN SAMHERI,"Pasal 114, Pasal 609, pasal 240, pasal 609, ...",bahwa terdakwa telah mengajukan permintaan ban...
1,putusan 8,296/PID.SUS/2026/PT,2026,Pengadilan Tinggi Palembang,Pidana Khusus,ISKANDAR AK BIN ABU KOSIM,"Pasal\n609, Pasal 114, Pasal 25, Pasal 609, pa...",bahwa pada tanggal 23 April 2026 Penuntut Umum...
2,putusan 28,99/Pid,2019,Pengadilan Tinggi Medan,Pidana Khusus,Tidak ditemukan,"Pasal\n112, Pasal\n114, Pasal 112, Pasal 127, ...","Menimbang, bahwa Terdakwa-Terdakwa diajukan ke..."
3,putusan 20,53/,2021,Pengadilan Tinggi Ambon,Pidana Khusus,Tidak ditemukan,"Pasal\n127, Pasal 112, Pasal 127, Pasal 112,...",Bahwa Ia Terdakwa JAMAL TUAKIA Alias MALO Alia...
4,putusan 18,2281,2016,Pengadilan \nTinggi\nYogyakarta,Pidana Khusus,Tidak ditemukan,"Pasal\n112, Pasal\n114, Pasal\n13, Pasal\n253,...",Bahwa Terdakwa Abdul Azis alias Andi bin H. Su...


Cell 13 - Cek Missing Value

In [20]:
df[
    [
        "nomor_putusan",
        "tahun",
        "pengadilan",
        "jenis_perkara",
        "pihak",
        "pasal",
        "ringkasan_fakta"
    ]
].isnull().sum()

,0
nomor_putusan,0
tahun,0
pengadilan,0
jenis_perkara,0
pihak,0
pasal,0
ringkasan_fakta,0


Cell 14 - Simpan Dataset Hasil

In [21]:
df.to_csv(
    "/content/CBR-Legal-Case/data/processed/cases.csv",
    index=False
)

print("Metadata berhasil disimpan")

Metadata berhasil disimpan


Cell 15 - Kesimpulan

## Hasil Tahap Case Representation

Metadata yang berhasil diekstraksi:

- Nomor Putusan
- Tahun
- Pengadilan
- Jenis Perkara
- Pihak
- Pasal
- Ringkasan Fakta

Representasi kasus ini digunakan sebagai dasar pada tahap retrieval menggunakan TF-IDF dan Cosine Similarity.